# ClarifyRL — GRPO Training with TRL + Unsloth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agarwalanu3103/clarify-rl/blob/main/training/train_grpo.ipynb)

Train **Qwen2.5-1.5B-Instruct** (4-bit via Unsloth) to ask clarifying questions before acting, using **GRPO** (Group Relative Policy Optimization) from TRL.

The model interacts with the ClarifyRL environment (hosted on HF Spaces) through the `environment_factory` pattern — TRL handles rollouts, tool-call parsing, and reward collection automatically.

## Compute requirements
- **GPU**: T4 16GB (Colab free tier or HF Jobs `t4-small`)
- **VRAM**: ~6GB with Unsloth 4-bit QLoRA
- **Time**: ~15min for 100 steps (smoke test), ~90min for 600 steps (full run)

In [ ]:
!pip install -Uq "trl[vllm]" unsloth trackio requests

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Configuration

In [ ]:
import random

ENV_BASE_URL = "https://agarwalanu3103-clarify-rl.hf.space"
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "clarify-rl-grpo-qwen2.5-1.5b"
MAX_STEPS = 100  # bump to 600 for full run
SEED = 42

DIFFICULTIES = ["easy", "medium", "hard"]
DIFFICULTY_WEIGHTS = [0.5, 0.3, 0.2]

random.seed(SEED)

prompt = """You are a helpful assistant that books and plans things for users.
When you receive a request, you may not have all the information needed.
You can:
1. ASK clarifying questions using the ask_question tool (max 6 total)
2. PROPOSE a final plan using the propose_plan tool when you have enough info
3. GET the task description again using the get_task_info tool

Be efficient: ask only what you NEED, then propose a plan.
Do NOT include preferences in the plan that you weren't told about."""

## Define the ClarifyRL Environment

The `ClarifyEnv` class wraps our HF Space into the interface expected by TRL's `environment_factory`.

Public methods with docstrings are automatically exposed as tools:
- `ask_question(question)` — ask a clarifying question
- `propose_plan(plan)` — submit a final plan as JSON
- `get_task_info()` — retrieve the task description

In [ ]:
import json
import requests


class ClarifyEnv:
    def __init__(self):
        self.base_url = ENV_BASE_URL
        self.reward = 0.0
        self.done = False
        self._session = requests.Session()

    def reset(self, **kwargs) -> str | None:
        task_id = random.choices(DIFFICULTIES, weights=DIFFICULTY_WEIGHTS, k=1)[0]
        resp = self._session.post(
            f"{self.base_url}/reset",
            json={"task_id": task_id},
            timeout=30,
        )
        data = resp.json()
        obs = data.get("observation", {})
        self.reward = data.get("reward", 0.0)
        self.done = data.get("done", False)
        result = obs.get("result", {})
        if isinstance(result, str):
            result = json.loads(result)
        request_text = result.get("request", "")
        family = result.get("family", "")
        max_steps = result.get("max_steps", 10)
        return (
            f"USER REQUEST: {request_text}\n"
            f"Task family: {family}\n"
            f"You have {max_steps} steps. Ask clarifying questions, then propose a plan."
        )

    def _step(self, tool_name: str, arguments: dict) -> dict:
        action = {
            "type": "call_tool",
            "tool_name": tool_name,
            "arguments": arguments,
        }
        resp = self._session.post(
            f"{self.base_url}/step",
            json=action,
            timeout=30,
        )
        data = resp.json()
        obs = data.get("observation", {})
        self.reward = data.get("reward", 0.0)
        self.done = data.get("done", False)
        result = obs.get("result", {})
        if isinstance(result, dict):
            return json.dumps(result)
        return str(result)

    def ask_question(self, question: str) -> str:
        """
        Ask a clarifying question to gather missing information from the user.

        Args:
            question: The clarifying question to ask.

        Returns:
            The user's response to your question.
        """
        return self._step("ask_question", {"question": question})

    def propose_plan(self, plan: str) -> str:
        """
        Submit your final plan as a JSON string with the required fields.

        Args:
            plan: A JSON string containing the plan fields you've gathered.

        Returns:
            The scoring result and episode outcome.
        """
        return self._step("propose_plan", {"plan": plan})

    def get_task_info(self) -> str:
        """
        Retrieve the current task description and requirements.

        Returns:
            The task information including the user request and instructions.
        """
        return self._step("get_task_info", {})

## Smoke test the environment

In [ ]:
env = ClarifyEnv()
obs = env.reset()
print("Initial observation:", obs)
print()

resp = env.ask_question("What type of backend do you need?")
print("Q1 response:", resp)
print("Reward:", env.reward, "Done:", env.done)
print()

resp = env.propose_plan('{"language": "Python", "framework": "FastAPI"}')
print("Plan response:", resp)
print("Reward:", env.reward, "Done:", env.done)

## Define the reward function

The reward function reads the cumulative reward tracked by each `ClarifyEnv` instance after the episode completes.

In [ ]:
def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]

## Create the dataset

Each entry triggers one rollout episode during training. We use 3000 episodes (can be reduced for smoke tests). The prompt is formatted as a chat message.

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "prompt": [[{"role": "user", "content": prompt}] for _ in range(3000)]
})

## GRPO Config

Using vLLM colocated mode for fast generation on a single T4. Gradient checkpointing keeps memory low.

In [ ]:
from trl import GRPOConfig

grpo_config = GRPOConfig(
    # Training schedule
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=16,
    per_device_train_batch_size=1,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    optim="adamw_torch",
    max_grad_norm=1.0,

    # GRPO
    num_generations=2,
    max_completion_length=1024,
    log_completions=True,
    num_completions_to_print=2,
    beta=0.04,

    # vLLM
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.15,
    vllm_max_model_length=3072,

    # Logging
    output_dir=OUTPUT_DIR,
    report_to="trackio",
    trackio_space_id=OUTPUT_DIR,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,

    # Memory
    gradient_checkpointing=True,

    # Hub
    push_to_hub=True,
)

## Create GRPOTrainer and train

The `environment_factory=ClarifyEnv` tells the trainer to automatically handle rollouts, tool-call parsing, and reward collection.

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=ClarifyEnv,
)

## GPU stats before training

In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

## Train!

In [ ]:
trainer_stats = trainer.train()

## Post-training stats

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_memory_percentage = round(used_memory_for_training / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {training_memory_percentage} %.")

## Save and push to Hub

In [ ]:
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub()

## Quick eval: run 5 episodes with trained model

Smoke test that the trained model produces better plans than baseline.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

fine_tuned_model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype="float16", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)


def run_eval_episode(model, tokenizer, difficulty="easy"):
    env = ClarifyEnv()
    # Override difficulty for controlled eval
    resp = env._session.post(f"{env.base_url}/reset", json={"task_id": difficulty}, timeout=30)
    data = resp.json()
    obs = data.get("observation", {})
    env.reward = data.get("reward", 0.0)
    env.done = data.get("done", False)
    result = obs.get("result", {})
    if isinstance(result, str):
        result = json.loads(result)
    request_text = result.get("request", "")
    family = result.get("family", "")

    initial_obs = f"USER REQUEST: {request_text}\nTask family: {family}"
    messages = [
        {"role": "user", "content": prompt},
        {"role": "user", "content": initial_obs},
    ]

    for turn in range(10):
        if env.done:
            break
        prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=512)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

        print(f"  Turn {turn+1}: {generated_text[:120]}...")

        # Parse tool call from generated text
        try:
            if "ask_question" in generated_text:
                start = generated_text.index("{")
                end = generated_text.rindex("}") + 1
                args = json.loads(generated_text[start:end])
                if "arguments" in args:
                    args = args["arguments"]
                feedback = env.ask_question(args.get("question", ""))
            elif "propose_plan" in generated_text:
                start = generated_text.index("{")
                end = generated_text.rindex("}") + 1
                args = json.loads(generated_text[start:end])
                if "arguments" in args:
                    args = args["arguments"]
                feedback = env.propose_plan(args.get("plan", "{}"))
            elif "get_task_info" in generated_text:
                feedback = env.get_task_info()
            else:
                feedback = "Could not parse tool call. Please use ask_question, propose_plan, or get_task_info."
            messages.append({"role": "assistant", "content": generated_text})
            messages.append({"role": "user", "content": feedback})
        except Exception as e:
            print(f"  Parse error: {e}")
            break

    print(f"  => Final reward: {env.reward}, Done: {env.done}")
    return env.reward


print("=== Post-training eval (5 episodes) ===")
rewards = []
for diff in ["easy", "easy", "medium", "medium", "hard"]:
    print(f"\n[{diff}]")
    r = run_eval_episode(fine_tuned_model, tokenizer, diff)
    rewards.append(r)
print(f"\nAverage reward: {sum(rewards)/len(rewards):.3f}")